In [2]:
# ---------------------------------------------------------------------------
# 1. РЕГУЛЯРНЫЕ ВЫРАЖЕНИЯ ДЛЯ УДАЛЕНИЯ ШУМА / ПЕРСОНАЛЬНЫХ ДАННЫХ
# ---------------------------------------------------------------------------

RE_IMG_TAG = re.compile(r"<\s*Картинка\s*>", re.IGNORECASE)

# Номера ЭМК: "ЭМК 15200760", "ЭМК №1032215", "ЭМК№ 123"
RE_EMK = re.compile(r"ЭМК\s*№?\s*\d+", re.IGNORECASE)

# Внутренние коды заявок вида W15-0470-0235.
# Без \b на конце: в сырых данных код иногда слипается с кириллицей без
# пробела ("W07-0600-0227Программное"), а \b не срабатывает на границе
# цифра-кириллица (обе считаются "словными" в Unicode-режиме).
RE_TICKET_CODE = re.compile(r"(?<![A-Za-zА-Яа-яЁё0-9])[wW]\d{2}-\d{4}-\d{4}")

# "тел. 7193", "т. 7418", "тел 7193" + сам номер
RE_PHONE_LABELED = re.compile(
    r"\b(тел|т)\.?\s*:?\s*\d{3,11}\b", re.IGNORECASE
)

# Отдельно стоящие длинные числа (внутренние/городские номера, коды) —
# применяем ПОСЛЕ удаления дат, чтобы не задеть их.
RE_LONG_NUMBER = re.compile(r"(?<!\d)\d{5,11}(?!\d)")

# Даты: 03.01.2026, 3.1.26, 03/01/2026, 21.10.1992г. (год иногда слит с "г.")
RE_DATE = re.compile(r"\b\d{1,2}[./]\d{1,2}[./]\d{2,4}(?:\s?г\.?)?", re.IGNORECASE)

# Время: 12:35:48, 9:59
RE_TIME = re.compile(r"\b\d{1,2}:\d{2}(?::\d{2})?\b")

# ФИО-паттерны: "Фамилия И.О." (Фадеевой О.Н., Мамедовой Регины Тахировны частично не ловится,
# но такие полные ФИО прописью оставляем — они не мешают семантике так сильно, как номера).
RE_FIO_INITIALS = re.compile(r"\b[А-ЯЁ][а-яё]+\s+[А-ЯЁ]\.\s?[А-ЯЁ]\.")

# Неразрывные пробелы и служебные символы
RE_NBSP = re.compile(r"[\xa0\u200b]")
RE_MULTI_SPACE = re.compile(r"[ \t]{2,}")
RE_MULTI_NEWLINE = re.compile(r"\n{2,}")

# ---------------------------------------------------------------------------
# 2. ШАБЛОННЫЕ ФРАЗЫ ОПЕРАТОРОВ (в поле "Решение" / "Тех. Решение")
#    Убираем вводные клише, которые не несут смысла для кластеризации.
# ---------------------------------------------------------------------------

BOILERPLATE_PATTERNS = [
    r"в\s*продолжени[ие]\s+телефонного\s+разговора,?\s*дублиру[юе]\s+информацию\.?",
    r"добр(ый|ой)\s+(день|ночи|утро)!?,?",
    r"обращение\s+закрыва[юе]\s+по\s+согласованию\s+с\s+пользователем\.?",
    r"по\s+согласованию\s+закрыва[юе]\s+обращение\.?",
    r"приносим\s+извинения\s+за\s+доставленн\w*\s+неудобств\w*\.?",
]
RE_BOILERPLATE = re.compile("|".join(BOILERPLATE_PATTERNS), re.IGNORECASE)

# Без \b на границах: GUID иногда встречается внутри путей/URL, слипаясь с "_"
# или другим словным символом, а \b не срабатывает на границе digit/hex <-> "_".
RE_GUID = re.compile(
    r"(?<![0-9a-fA-F-])[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}(?![0-9a-fA-F])"
)
RE_SNILS = re.compile(r"\b\d{3}-\d{3}-\d{3}\s?\d{2}\b")
RE_EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
# Разбивка последних 7 цифр локального номера встречается в разных форматах
# (444-00-33, 444-0-333, 4440033 и т.п.) — не привязываемся к жёсткой схеме
# групп, просто разрешаем цифры вперемешку с дефисами/пробелами.
RE_MOBILE_PHONE = re.compile(r"\+?7\s*\(?\d{3}\)?[\s\-]*\d[\d\-\s]{4,8}\d")

# Ценные для темы обращения ярлыки анкеты ("СНИЛС:", "Табельный номер:" и т.п.)
# несут сигнал "это заявка на УЗ/доступ" — оставляем сам ярлык, вырезаем только
# значение после двоеточия до конца строки.
RE_FIELD_VALUE = re.compile(
    r"(СНИЛС|Табельный номер|Дата рождения|Мобильный телефон|Электронный адрес|"
    r"Учетная запись пользователя компьютера \(Active Directory\)|GUID)"
    r"\s*:\s*[^\n]*",
    re.IGNORECASE,
)

# Полные ФИО без инициалов: "Гончуков Сергей Андреевич" (Фамилия Имя Отчество)
RE_FIO_FULL = re.compile(
    r"\b[А-ЯЁ][а-яё]+\s+[А-ЯЁ][а-яё]+\s+[А-ЯЁ][а-яё]+(?:ич|вич|на|вна)\b"
)

MAX_DESC_LEN = 3000  # символов; обрезаем длинные автосгенерированные тексты до очистки

In [3]:
def clean_text(text: str, strip_boilerplate: bool = False) -> str:
    """Очистка текста под новую схему: сначала общие паттерны (v1), затем новые."""
    if not isinstance(text, str):
        return ""

    t = text[:MAX_DESC_LEN] if len(text) > MAX_DESC_LEN else text

    t = RE_NBSP.sub(" ", t)
    t = RE_IMG_TAG.sub(" ", t)
    t = RE_FIELD_VALUE.sub(lambda m: m.group(1) + ":", t)  # значение анкеты -> оставляем только ярлык
    t = RE_GUID.sub(" ", t)
    t = RE_SNILS.sub(" ", t)
    t = RE_EMAIL.sub(" ", t)
    t = RE_MOBILE_PHONE.sub(" ", t)
    t = RE_EMK.sub(" ", t)
    # применяем дважды: коды/GUID иногда слипаются друг с другом без разделителя
    # ("w15-0470-0903w15-0470-0903"), и первый проход вскрывает второе вхождение
    for _ in range(2):
        t = RE_TICKET_CODE.sub(" ", t)
    t = RE_PHONE_LABELED.sub(" ", t)
    t = RE_DATE.sub(" ", t)
    t = RE_TIME.sub(" ", t)
    t = RE_LONG_NUMBER.sub(" ", t)
    t = RE_FIO_FULL.sub(" ", t)
    t = RE_FIO_INITIALS.sub(" ", t)

    if strip_boilerplate:
        t = RE_BOILERPLATE.sub(" ", t)

    t = RE_MULTI_NEWLINE.sub("\n", t)
    t = RE_MULTI_SPACE.sub(" ", t)
    t = "\n".join(line.strip() for line in t.split("\n") if line.strip())
    return t.strip()

In [4]:
def load_and_clean(path: str) -> pd.DataFrame:
    """
    Загружает CSV новой схемы и возвращает нормализованный DataFrame.

    Устойчив к вариациям схемы между выгрузками: колонки "Обращение.Решение"
    и "Обращение.Массовый_случай" опциональны — используются, если присутствуют
    в файле, иначе пропускаются без ошибки. Это позволяет одним и тем же
    скриптом обрабатывать как более ранние выгрузки (без этих полей), так и
    более новые (с ними), без ручной правки кода под каждый экспорт.
    """
    raw = pd.read_csv(path, sep=";", quotechar='"', encoding="utf-8")
    raw = raw.rename(columns=lambda c: c.strip())
    raw = raw.rename(columns={
        "Обращение.Номер": "ticket_id",
        "Обращение.Дата": "date",
        "Клиент.Категория": "client_category",
        "Обращение.Группа": "group_raw",
        "Обращение.Группа_действие": "group_action_raw",
        "Обращение.Тип": "type_raw",
        "Обращение.Тема": "subject_raw",
        "Обращение.Описание": "description_raw",
        "Обращение.Решение": "solution_raw",
        "Обращение.Массовый_случай": "mass_case",
    })

    has_solution = "solution_raw" in raw.columns
    has_mass_case = "mass_case" in raw.columns
    print(f"  Колонка 'Обращение.Решение': {'есть' if has_solution else 'отсутствует'} в этой выгрузке")
    print(f"  Колонка 'Обращение.Массовый_случай': {'есть' if has_mass_case else 'отсутствует'} в этой выгрузке")

    n_rows = len(raw)
    if not raw["ticket_id"].is_unique:
        dupes = raw["ticket_id"].duplicated().sum()
        print(f"  !! ВНИМАНИЕ: 'Обращение.Номер' не уникален ({dupes} повторов) — "
              f"ожидалось, что в этой выгрузке номер уникален. Проверьте источник данных.")

    raw["date"] = pd.to_datetime(raw["date"], errors="coerce")

    # категориальные поля: подчистить пробелы, заполнить пропуски явной меткой.
    # fillna() ПЕРЕД astype(str): иначе на нек-рых версиях pandas (string dtype
    # с NA) отсутствующее значение не превращается в текстовое "nan" и не
    # ловится последующим isin(["nan", ...]), а остаётся NA/float и ломает
    # дальнейшую сортировку/сравнение строк.
    for col in ["group_raw", "group_action_raw", "type_raw"]:
        raw[col] = raw[col].fillna("не указано").astype(str).str.strip()
        raw.loc[raw[col].isin(["nan", "None", ""]), col] = "не указано"

    desc_len_before = raw["description_raw"].fillna("").astype(str).str.len()
    raw["description_truncated"] = desc_len_before > MAX_DESC_LEN

    raw["subject_clean"] = raw["subject_raw"].apply(clean_text_v2)
    raw["description_clean"] = raw["description_raw"].apply(clean_text_v2)
    if has_solution:
        raw["solution_clean"] = raw["solution_raw"].apply(lambda x: clean_text_v2(x, strip_boilerplate=True))
    else:
        raw["solution_clean"] = ""

    # текст для кластеризации: категориальные ярлыки системы + тема + описание + решение.
    # Категориальные поля добавляются как есть (без сокращения) — это НЕ шум,
    # а полезный сигнал, который направит кластеризацию точнее свободного текста.
    def build_text(row):
        parts = [
            f"Группа: {row['group_raw']}.",
            f"Действие: {row['group_action_raw']}.",
            f"Тип обращения: {row['type_raw']}.",
            row["subject_clean"],
            row["description_clean"],
            row["solution_clean"],
        ]
        skip = {"Группа: не указано.", "Действие: не указано.", "Тип обращения: не указано."}
        return "\n".join(p for p in parts if p and p not in skip)

    raw["text_for_clustering"] = raw.apply(build_text, axis=1)
    raw["text_len"] = raw["text_for_clustering"].str.len()

    n_empty_desc = raw["description_raw"].isna().sum()
    n_empty_final = (raw["text_for_clustering"].str.strip() == "").sum()

    raw.attrs["n_rows"] = n_rows
    raw.attrs["n_empty_description_raw"] = int(n_empty_desc)
    raw.attrs["n_empty_final_text"] = int(n_empty_final)
    raw.attrs["n_truncated"] = int(raw["description_truncated"].sum())
    raw.attrs["has_solution"] = has_solution
    raw.attrs["has_mass_case"] = has_mass_case
    raw.attrs["groups_present"] = sorted(raw["group_raw"].unique().tolist())

    # Excel допускает не более 32767 символов в ячейке — у нас есть описания
    # длиной до ~500 000 (автосгенерированные уведомления мониторинга).
    # Обрезаем ТОЛЬКО копию для сохранения в файл; description_truncated уже
    # посчитан по исходной длине и не меняется.
    EXCEL_CELL_LIMIT = 32000
    raw["description_raw"] = raw["description_raw"].astype(str).str.slice(0, EXCEL_CELL_LIMIT)
    if has_solution:
        raw["solution_raw"] = raw["solution_raw"].astype(str).str.slice(0, EXCEL_CELL_LIMIT)

    cols = [
        "ticket_id", "date", "client_category",
        "group_raw", "group_action_raw", "type_raw",
        "subject_raw", "description_raw", "description_truncated",
    ]
    if has_solution:
        cols += ["solution_raw"]
    if has_mass_case:
        cols += ["mass_case"]
    cols += [
        "subject_clean", "description_clean", "solution_clean",
        "text_for_clustering", "text_len",
    ]
    return raw[cols]

In [ ]:
src = 'obraschenie.csv'
dst = 'Medialog_Clustered.xlsx'

result = load_and_clean_v2(src)
result.to_excel(dst, index=False)
print(f"Готово: {len(result)} строк сохранено в {dst}")
print(f"  Исходных строк: {result.attrs['n_rows']}")
print(f"  Пустых Описание в исходнике: {result.attrs['n_empty_description_raw']}")
print(f"  Итоговых пустых text_for_clustering: {result.attrs['n_empty_final_text']}")
print(f"  Обрезано длинных описаний (>{MAX_DESC_LEN} симв.): {result.attrs['n_truncated']}")
print(f"  Уникальных значений 'Группа' в этой выгрузке: {len(result.attrs['groups_present'])}")